# ApprovalMax v2 Dashboard Table Refresh


In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

catalog = 'approvalmax_ai_platform'
schemas = {
    'bronze': 'bronze',
    'silver': 'silver',
    'vault': 'vault',
    'gold': 'gold',
    'monitoring': 'monitoring',
}
spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalog}')
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schemas['monitoring']}")

def full(schema, table):
    return f'{catalog}.{schema}.{table}'

def table_exists(table_name):
    try:
        spark.table(table_name).limit(1).count()
        return True
    except Exception:
        return False

def save(df, table_name):
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(table_name)

monitoring = schemas['monitoring']
gold_lifecycle = full(schemas['gold'], 'fact_approval_document_lifecycle')
audit_table = full(monitoring, 'etl_audit_log')
gx_table = full(monitoring, 'great_expectations_results')

row_count_rows = []
for layer, schema in schemas.items():
    if layer == 'monitoring':
        continue
    for row in spark.sql(f'SHOW TABLES IN {catalog}.{schema}').collect():
        table_name = full(schema, row.tableName)
        row_count_rows.append({'layer': layer, 'table_name': table_name, 'row_count': spark.table(table_name).count(), 'refreshed_at': datetime.now(timezone.utc)})

save(spark.createDataFrame(row_count_rows), full(monitoring, 'dashboard_layer_row_counts'))

if table_exists(audit_table):
    audit = spark.table(audit_table)
    save(audit, full(monitoring, 'dashboard_etl_step_timeline'))
    save(audit.groupBy('run_id').agg(F.min('started_at').alias('started_at'), F.max('ended_at').alias('ended_at'), F.count('*').alias('step_count'), F.sum(F.when(F.col('status') == 'failed', 1).otherwise(0)).alias('failed_step_count')), full(monitoring, 'dashboard_pipeline_run_summary'))

if table_exists(gx_table):
    save(spark.table(gx_table), full(monitoring, 'dashboard_quality_status'))

if table_exists(gold_lifecycle):
    gold = spark.table(gold_lifecycle)
    save(gold, full(monitoring, 'dashboard_gold_documents'))
    kpis = gold.agg(F.count('*').alias('document_count'), F.sum('total_amount').alias('total_amount'), F.avg('approval_cycle_time_minutes').alias('avg_approval_cycle_time_minutes'), F.sum('approval_sla_breach_flag').alias('sla_breach_count')).withColumn('refreshed_at', F.current_timestamp())
    save(kpis, full(monitoring, 'dashboard_kpi_snapshot'))

print('Dashboard tables refreshed.')
